In [1]:
# import
import pandas as pd
import requests
import time
import json
import os
import re
import time
from tqdm import tqdm  # For progress bar, install with pip install tqdm if needed


In [3]:
# API configuration
API_DELAY = 1  # seconds between API calls to avoid rate limiting


def get_iocsANDexploits(cve_id):
    """Enhanced IOC extraction from multiple sources with better OTX parsing"""
    time.sleep(API_DELAY)
    try:
        # First try AlienVault OTX
        otx_url = f"https://otx.alienvault.com/api/v1/indicators/cve/{cve_id}"
        otx_response = requests.get(otx_url)
        otx_response.raise_for_status()
        otx_data = otx_response.json()
        
        iocs = {
            'IPs': set(),
            'Domains': set(),
            'URLs': set(),
            'Hashes': set(),
            'FileNames': set(),
            'Emails': set(),
            'Pulse_count': 0,
            'Pulse_samples': []
        }

        # Parse OTX pulses more thoroughly
        if 'pulse_info' in otx_data and 'pulses' in otx_data['pulse_info']:
            pulses = otx_data['pulse_info']['pulses']
            iocs['Pulse_count'] = len(pulses)
            
            # Sample first 3 pulse names for context
            iocs['Pulse_samples'] = [pulse.get('name', '') for pulse in pulses[:3]]
            
            # Try to get indicators from pulse details
            for pulse in pulses[:5]:  # Limit to first 5 pulses to avoid timeout
                try:
                    pulse_id = pulse['id']
                    pulse_detail_url = f"https://otx.alienvault.com/api/v1/pulses/{pulse_id}"
                    pulse_response = requests.get(pulse_detail_url, timeout=10)
                    pulse_data = pulse_response.json()
                    
                    if 'indicators' in pulse_data:
                        for indicator in pulse_data['indicators']:
                            indicator_type = indicator.get('type', '')
                            indicator_value = indicator.get('indicator', '')
                            
                            if indicator_type == 'IPv4':
                                iocs['IPs'].add(indicator_value)
                            elif indicator_type == 'domain':
                                iocs['Domains'].add(indicator_value)
                            elif indicator_type == 'URL':
                                iocs['URLs'].add(indicator_value)
                            elif indicator_type in ['FileHash-MD5', 'FileHash-SHA1', 'FileHash-SHA256']:
                                iocs['Hashes'].add(indicator_value)
                            elif indicator_type == 'filename':
                                iocs['FileNames'].add(indicator_value)
                            elif indicator_type == 'email':
                                iocs['Emails'].add(indicator_value)
                except Exception as pulse_error:
                    # print(f"Error processing pulse {pulse_id}: {str(pulse_error)}")
                    continue

        # If still no IOCs, try additional sources
        if not any(iocs.values()):  # If all IOC lists are empty
            # print(f"No IOCs found in OTX for {cve_id}, trying threatminer as additional sources")
            iocs.update(get_iocs_from_secondary_sources(cve_id))
        
        # Convert sets to lists
        for key in iocs:
            if isinstance(iocs[key], set):
                iocs[key] = list(iocs[key])
                
        return {
            'source': 'OTX alien vault / threatminer if alien vault empty',
            'Exploits': otx_response.json().get('exploits', []),
            'pulse_context': {
                'count': iocs.pop('Pulse_count'),
                'samples': iocs.pop('Pulse_samples')
            },
            'iocs': iocs
        }
        
    except Exception as e:
        # print(f"Error fetching IOCs for {cve_id}: {str(e)}")
        return None

def get_iocs_from_secondary_sources(cve_id):
    """Fallback IOC extraction from other sources"""
    secondary_iocs = {
        'Static_Item_Source:': "ThreatMiner",
        'IPs': set(),
        'Domains': set(),
        'URLs': set(),
        'Hashes': set()
    }
    
    try:
        # Try ThreatMiner
        tm_url = f"https://api.threatminer.org/v2/vulnerability.php?q={cve_id}&api=True&rt=1"
        tm_response = requests.get(tm_url, timeout=10)
        if tm_response.status_code == 200:
            for result in tm_response.json().get('results', []):
                if result.get('type') == 'domain':
                    secondary_iocs['Domains'].add(result['value'])
                elif result.get('type') == 'ip':
                    secondary_iocs['IPs'].add(result['value'])
    
    except Exception as e:
        # print(f"Error with ThreatMiner for {cve_id}: {str(e)}")
        pass
    
    return {k: list(v) for k, v in secondary_iocs.items() if v}

In [ ]:
get_iocsANDexploits('CVE-2017-5638')

In [ ]:
# MAIN

In [ ]:
# Find ways to integrate when it come from otx or data miner and find a way to avoid too many requiest to the API issue

# CVE to IOCs and exploits


# print(os.getcwd())
CVE_CWE_mappingDF = pd.read_csv('../CVE_CWE_mapping.csv', sep=',')

CVE_list = list(set(CVE_CWE_mappingDF['Key'].tolist()))  # Removes duplicates but loses original order

CVE_to_Exploit_N_IOC_dict = dict()

for CVE in CVE_list:
    CVE_to_Exploit_N_IOC_dict[CVE] = get_iocsANDexploits(CVE)

    
    

In [ ]:
# Find ways to integrate when it come from otx or data miner and find a way to avoid too many requiest to the API issue

# CVE to IOCs and exploits

# print(os.getcwd())
CVE_CWE_mappingDF = pd.read_csv('../CVE_CWE_mapping.csv', sep=',')

CVE_list = list(set(CVE_CWE_mappingDF['Key'].tolist()))  # Removes duplicates but loses original order

CVE_to_Exploit_N_IOC_dict = dict()

# Configuration for rate limiting
MAX_RETRIES = 3  # Number of retries for failed requests
RETRY_DELAY = 5  # Seconds to wait between retries
REQUEST_DELAY = 1  # Seconds between normal requests (from your API_DELAY)

# Process CVEs with rate limiting and error handling
for CVE in tqdm(CVE_list, desc="Processing CVEs"):
    retries = 0
    success = False
    
    while retries < MAX_RETRIES and not success:
        try:
            # Get IOCs and exploits with your original function
            result = get_iocsANDexploits(CVE)
            
            if result is not None:
                CVE_to_Exploit_N_IOC_dict[CVE] = result
                success = True
            else:
                print(f"Warning: No results for {CVE} (attempt {retries + 1})")
                
        except requests.exceptions.Timeout:
            print(f"Timeout for {CVE}, retrying... (attempt {retries + 1})")
            time.sleep(RETRY_DELAY)
            retries += 1
            
        except requests.exceptions.RequestException as e:
            print(f"Error for {CVE}: {str(e)}")
            time.sleep(RETRY_DELAY)
            retries += 1
            
        except Exception as e:
            print(f"Unexpected error for {CVE}: {str(e)}")
            time.sleep(RETRY_DELAY)
            retries += 1
    
    if not success:
        print(f"Failed to process {CVE} after {MAX_RETRIES} attempts")
        CVE_to_Exploit_N_IOC_dict[CVE] = None  # Store None for failed CVEs
    
    # Respect rate limiting between successful requests
    if success and CVE != CVE_list[-1]:  # Don't wait after last item
        time.sleep(REQUEST_DELAY)

Processing CVEs:   0%|                                                                                       | 32/75287 [01:32<51:09:40,  2.45s/it]

In [ ]:
V2 = CVE_to_Exploit_N_IOC_dict

